# 🤖 Workshop Day 2: Building Agents — Agent Google ADK

**Agentic RAG Workshop** | Day 2 of 3

---

### 🎯 
1. **Agent ** — Chatbot 
2. **Google ADK** — Agent Google
3. **Tool** — Agent "" 
4. **RAG Tool** — Agent VectorDB Day 1
5. **Multi-Agent** — Agent 
6. **Agentic RAG** — + + 

### 🗺️ Pipeline

| Day 1 (Data Engineering) | ➡️ | Day 2 (Agent Building) |
|---|---|---|
| Raw Data → Dedup → Chunk | | 2.1 Agent |
| → Embed | | 2.2 Tool |
| → Index (Qdrant) | ➡️ | 2.3 RAG Tool + Qdrant ⭐ |
| → Retrieve | | 2.4 Multi-Agent |
| | | 2.5 Session & Memory |
| | | 2.6 Agentic RAG ⭐ |

## 📦 Section 0: Dependencies

> ⚠️ Google ADK terminal `adk web` Colab 

### 🆚 ADK vs LangChain vs CrewAI

| | **Google ADK** | **LangChain** | **CrewAI** |
|---|:---:|:---:|:---:|
| **** | Multi-Agent + Gemini | LLM+Tool | Agent |
| **LLM** | Gemini () + | LLM | LLM |
| **Multi-Agent** | ✅ Built-in | ❌ | ✅ Built-in |
| **Workflow** | Sequential/Parallel/Loop | LCEL Chain | Sequential/Hierarchical |
| **Dev UI** | ✅ `adk web` | ❌ | ❌ |
| **** | Google | LangChain Inc. | CrewAI Inc. |

> 💡 **ADK** framework Google Multi-Agent 
> **Gemini** 

In [ ]:
%%time
# Google ADK libraries Day 1
!pip install -q google-adk \
 google-genai \
 sentence-transformers \
 qdrant-client \
 langchain-text-splitters

print('✅ !')

In [ ]:
%%time
# Gemini API Key
import os

try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 api_key = input('🔑 Gemini API Key: ')
 os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

# API
from google import genai
try:
 client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=' ')
 print(f'✅ API : {(resp.text.strip() if resp.text else '( output)')}')
except Exception as e:
 print(f'❌ API Error: {e}')
 print('💡 : 1) API Key ? 2) internet? 3) Free tier ?')

### 📚 Gemini API Parameters

 Gemini ( ADK) parameter :

#### 🎛️ Generation Parameters

| Parameter | Default | | |
|-----------|:-----------:|------|----------|
| `temperature` | 1.0 | 0.0 - 2.0 | "" |
| `top_p` | 0.95 | 0.0 - 1.0 | (nucleus sampling) |
| `top_k` | 40 | 1 - ∞ | step |
| `max_output_tokens` | 8192 | 1 - 65536 | |

#### 🌡️ Temperature — 

```
temperature = 0.0 → (deterministic)
 : , , RAG

temperature = 1.0 → 
 : chatbot , Agent

temperature = 2.0 → "" 
 : brainstorm, 
```

#### 🔍 top_p vs top_k

| | top_k | top_p |
|---|------|------|
| **** | k | probability ≤ p |
| **** | top_k=3 → 3 | top_p=0.9 → 90% |
| **** | → conservative | → conservative |

#### 🏷️ Gemini Models 

| Model | | | |
|-------|:--------:|:----:|----------|
| `gemini-2.5-flash` | ⚡ | 💰 | Agent, Tool calling, RAG |
| `gemini-2.5-pro` | 🐢 | 💰💰 | , |
| `gemini-3.1-flash` | ⚡⚡ | 💰 | Agent , |

> 💡 Workshop **`gemini-2.5-flash`** Tool calling 

In [ ]:
%%time
# ─── Temperature ───
from google import genai

prompt = ' AI 1 '

print('🧪 Temperature ( 3 )\n')

for temp in [0.0, 1.0, 2.0]:
 print(f'--- temperature = {temp} ---')
 for round_num in range(1, 4):
 try:
 resp = client.models.generate_content(
 model='gemini-2.5-flash',
 contents=prompt,
 config=genai.types.GenerateContentConfig(
 temperature=temp,
 max_output_tokens=100,
 )
 )
 print(f' {round_num}: {(resp.text.strip() if resp.text else '( output)')[:80]}')
 except Exception as e:
 print(f' ❌ Error: {e}')
 print()

print('💡 : temp=0.0 → , temp=2.0 → ')

In [ ]:
%%time
# ─── max_output_tokens ───
print('🧪 max_output_tokens\n')

for max_tokens in [20, 100, 500]:
 try:
 resp = client.models.generate_content(
 model='gemini-2.5-flash',
 contents=' AI ',
 config=genai.types.GenerateContentConfig(
 max_output_tokens=max_tokens,
 temperature=0.5
 )
 )
 text = resp.text.strip() if resp.text else '( output — token )'
 print(f'max_tokens={max_tokens:>3}: ({len(text)} chars) {text[:80]}...')
 except Exception as e:
 print(f' ❌ Error: {e}')

print('\n💡 max_output_tokens "" ""')

### 💡 
- `temperature=0.0` → **** — RAG 
- `temperature=2.0` → **** — creative
- `max_output_tokens` → ""

> 🎯 **Agent/RAG** : `temperature=0.5-1.0`, `max_output_tokens=2048+`

---
## 🤖 Section 2.1: Agent 

### Agent vs Chatbot — ?

| | Chatbot | Agent |
|---|---------|-------|
| **** | | + |
| **** | | , , API |
| **** | | |
| **** | FAQ Bot | ChatGPT + Plugins, Gemini |

### Google ADK ?

**Agent Development Kit (ADK)** = open-source Python framework Google

```
Agent = Model (LLM) + Instructions () + Tools ()
```

- **Gemini** LLM ( model )
- **Multi-Agent** 
- **Web UI** debug

In [ ]:
%%time
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

import warnings
warnings.filterwarnings('ignore')

# ─── Agent : Tool Instructions ───
greeting_agent = Agent(
 name='greeting_agent',
 model='gemini-2.5-flash',
 description='Agent ',
 instruction=""" AI Assistant 
 - 
 - 2 
 - emoji 
 """
)

print('✅ Agent !')
print(f' : {greeting_agent.name}')
print(f' Model: {greeting_agent.model}')

### 🔧 Async + Runner?

- **`async/await`** — ADK asynchronous Agent LLM (I/O bound)
- **`InMemoryRunner`** — Agent event loop 
- **`InMemorySessionService`** — RAM

```
Runner → Session → Agent → Agent 
 → Tool () → → Event 
```

> 💡 Colab `await` cell ( `asyncio.run()`)

In [ ]:
# ─── Agent Colab ───
import asyncio

async def chat_with_agent(runner, message, user_id='user_1', session_id='default_session'):
 """ Agent """
 content = types.Content(
 role='user',
 parts=[types.Part.from_text(text=message)]
 )
 
 final_response = ''
 tool_calls = []
 
 async for event in runner.run_async(
 user_id=user_id,
 session_id=session_id,
 new_message=content
 ):
 if event.content and event.content.parts:
 for part in event.content.parts:
 if part.text:
 final_response = part.text
 if part.function_call:
 tool_calls.append({
 'name': part.function_call.name,
 'args': dict(part.function_call.args) if part.function_call.args else {}
 })
 
 if tool_calls:
 print(f' 🔧 Tools : {[t["name"] for t in tool_calls]}')
 
 return final_response


async def demo_chat(agent, messages):
 """Demo Agent ( session )"""
 runner = InMemoryRunner(agent=agent, app_name=agent.name)
 session_id = f'session_{id(agent)}'
 
 # session runner
 await runner.session_service.create_session(
 app_name=agent.name,
 user_id='user_1',
 session_id=session_id
 )
 
 for msg in messages:
 print(f'\n👤 : {msg}')
 response = await chat_with_agent(runner, msg, session_id=session_id)
 print(f'🤖 Agent: {response}')


print('✅ chat ')

In [ ]:
# ─── Agent ───
await demo_chat(greeting_agent, [
 '',
 'AI ',
 ''
])

### 💡 
- Agent ** instruction** (, , emoji)
- instruction → Agent 
- Agent Tool → ** LLM** 

> 🎯 **Instruction Agent** — = Agent 

### 🧪 Exercise 2.1

Try changing the Agent `instruction` to different personas:
- An AI instructor
- A doctor giving health advice
- A financial consultant



In [ ]:
# 🧪 : instruction Agent 
# my_agent = Agent(
# name='...',
# model='gemini-2.5-flash',
# description='...',
# instruction='...'
# )
# await demo_chat(my_agent, ['', ''])

### ✍️ Tips: Instruction 

**Instruction = Agent** — 

| | |
|--------|--------|
| ** Role** | "" |
| ** Tool** | " calculate_bmi " |
| **** | " → " |
| ** Format** | " bullet points 5 " |
| ** Constraint** | " " |
| **** | " emoji" |

```python
# ❌ Instruction 
instruction = ""

# ✅ Instruction 
instruction = """ 
- calculate_bmi BMI 
- → tool
- 3 
- """
```

---
## 🔧 Section 2.2: Tool Agent

### Tool ?

**Tool = Python function Agent **

```
 "BMI 70 kg 175 cm"
 ↓
Agent → : tool calculate_bmi!
 ↓
Agent : calculate_bmi(weight_kg=70, height_cm=175)
 ↓
Agent → 
```

### 📌 Tool

| | ? |
|-----|-------|
| **docstring** | LLM docstring tool |
| **type hints** | LLM parameter |
| return **dict** | Agent dict |
| | LLM tool |

In [ ]:
%%time
# ─── Tool 1: BMI ───
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
 """ (BMI) 

 Args:
 weight_kg: 
 height_cm: 

 Returns:
 dict: BMI, , 
 """
 height_m = height_cm / 100
 bmi = weight_kg / (height_m ** 2)

 if bmi < 18.5:
 level = ''
 advice = ''
 elif bmi < 25:
 level = ''
 advice = ' '
 elif bmi < 30:
 level = ''
 advice = ''
 else:
 level = ''
 advice = ''

 return {'bmi': round(bmi, 1), 'level': level, 'advice': advice}


# ─── Tool 2: ───
def convert_temperature(value: float, from_unit: str) -> dict:
 """ Celsius Fahrenheit

 Args:
 value: 
 from_unit: ('celsius' 'fahrenheit')

 Returns:
 dict: 
 """
 if from_unit.lower() == 'celsius':
 result = (value * 9/5) + 32
 return {'result': round(result, 1), 'from': f'{value}°C', 'to': f'{result:.1f}°F'}
 elif from_unit.lower() == 'fahrenheit':
 result = (value - 32) * 5/9
 return {'result': round(result, 1), 'from': f'{value}°F', 'to': f'{result:.1f}°C'}
 else:
 return {'error': f' {from_unit}'}


# ─── Agent Tools ───
health_agent = Agent(
 name='health_agent',
 model='gemini-2.5-flash',
 description='Agent BMI ',
 instruction=""" Agent 
 - calculate_bmi BMI 
 - convert_temperature 
 - tools 
 """,
 tools=[calculate_bmi, convert_temperature]
)

print('✅ Health Agent 2 tools')
print(f' 🔧 Tools: {[t.__name__ for t in health_agent.tools]}')

In [ ]:
runner = InMemoryRunner(agent=health_agent, app_name=health_agent.name)
await runner.session_service.create_session(app_name=health_agent.name, user_id='user_1', session_id='test_session')
# ─── Tool Selection ───
print('═' * 60)
print('🧪 : Agent Tool ?')
print('═' * 60)

test_messages = [
 ' 70 kg 175 cm BMI ',
 '37 ',
 'AI ', # tool!
]

for msg in test_messages:
 print(f'\n👤 : {msg}')
 response = await chat_with_agent(runner, msg, session_id='test_session')
 print(f'🤖 Agent: {response}')

### 💡 
- BMI → Agent `calculate_bmi` 🔧
- → Agent `convert_temperature` 🔧
- → Agent ** tool** 💬

> Agent **** tool — chatbot!

### 🧪 Exercise 2.2

Create one custom tool and add it to the Agent:
- Calculate grade from score (`score → grade`)
- Calculate interest
- Convert currency
- Anything else you can think of!



In [ ]:
# 🧪 : tool 
# def my_tool(param1: float, param2: str) -> dict:
# """ tool (! LLM )"""
# ...
# return {...}
#
# my_agent = Agent(
# name='my_agent', model='gemini-2.5-flash',
# tools=[my_tool, calculate_bmi]
# )
# await demo_chat(my_agent, [' tool '])

---
## 🔍 Section 2.3: RAG Tool — Agent Qdrant ⭐

### Agentic RAG ?

```
RAG : → → ( )
Agentic RAG: → Agent → ? → ()
```

### Agentic RAG ?

| | RAG | Agentic RAG |
|---|----------|-------------|
| "" | VectorDB () | ✅ |
| "AI " | VectorDB ✅ | VectorDB ✅ |
| follow-up | | + ✅ |
| tool | ❌ | ✅ tool |

### Real-World
- **KADE ()**: Agent // KB 
- ** Study Bot**: Agent "" → , "" → 

> Agent **** ****

In [ ]:
%%time
# ─── Qdrant VectorDB ( Day 1) ───
import hashlib, random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# Embedding Model
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print('✅ embedding model')

# Day 1
documents = {
 'healthcare': ' AI X-ray 30 5 Deep Learning 95% ',
 'banking': ' KADE AI Assistant RAG 5 30 ',
 'education': ' RAG - PDF 500 vector embeddings multilingual model ',
 'agriculture': ' AI Computer Vision 92% 40%',
 'kmitl': ' (KMITL) AI Smart Campus IoT sensor Machine Learning 25% NLP RAG AI Tutor ',
 'nlp': ' NLP Word Segmentation PyThaiNLP tokenization',
 'rag_architecture': ' RAG 3 Retriever VectorDB, Reader , Generator RAG hallucination LLM ',
 'embedding': 'Text Embedding vector multilingual-e5-large Microsoft 100 vector 1024 vector embedding',
 'vector_db': 'Qdrant Vector Database ANN (Approximate Nearest Neighbor) vector Cosine Similarity, Dot Product, Euclidean Distance',
}

# Chunk + Embed + Index
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

all_chunks = []
all_sources = []
for source, text in documents.items():
 chunks = splitter.split_text(text)
 for chunk in chunks:
 all_chunks.append(chunk)
 all_sources.append(source)

print(f'📊 {len(all_chunks)} chunks {len(documents)} ')

# Embed
passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]
print(f'✅ Embedding: {embeddings.shape}')

# Create Qdrant collection
qdrant = QdrantClient(':memory:')
qdrant.create_collection(
 collection_name='thai_ai_kb',
 vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

# Upsert
points = [
 models.PointStruct(
 id=i,
 vector=embeddings[i].tolist(),
 payload={'text': all_chunks[i], 'source': all_sources[i]}
 )
 for i in range(len(all_chunks))
]
qdrant.upsert(collection_name='thai_ai_kb', points=points)
print(f'✅ Upsert {len(points)} points Qdrant')

In [ ]:
%%time
# ─── RAG Tool ───
def search_thai_ai_knowledge(query: str) -> dict:
 """ AI 

 tool :
 - AI ( )
 - RAG, Embedding, Vector Database
 - NLP 

 Args:
 query: 

 Returns:
 dict: 
 """
 query_vector = embed_model.encode(f'query: {query}').tolist()
 results = qdrant.query_points(
 collection_name='thai_ai_kb',
 query=query_vector,
 limit=3
 ).points

 if not results:
 return {'status': 'no_results', 'message': ''}

 contexts = []
 for r in results:
 contexts.append({
 'text': r.payload['text'],
 'source': r.payload['source'],
 'score': round(r.score, 4)
 })

 return {
 'status': 'success',
 'num_results': len(contexts),
 'results': contexts
 }

print('✅ RAG Tool ')

# tool 
test = search_thai_ai_knowledge('AI ')
print(f'\n🧪 : {test["num_results"]} ')
for r in test['results']:
 print(f' [{r["source"]}] score={r["score"]} | {r["text"][:60]}...')

In [ ]:
%%time
# ─── RAG Agent ───
rag_agent = Agent(
 name='thai_ai_expert',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=""" AI 

 :
 1. AI , case study, RAG
 → search_thai_ai_knowledge 
 2. 
 3. ""
 4. → 
 5. bullet points
 """,
 tools=[search_thai_ai_knowledge]
)

print('✅ RAG Agent !')

In [ ]:
# ─── Demo: Agent ───
print('═' * 60)
print('🧪 Demo: Agent ')
print('═' * 60)

await demo_chat(rag_agent, [
 '', # → 
 'AI ', # → Qdrant
 'RAG ', # → Qdrant
 '', # → 
])

### 💡 
- "" → Agent ** tool** 💬
- "AI " → Agent **** `search_thai_ai_knowledge` 🔧
- **** Qdrant 

> **Agentic RAG**: Agent → → 

### 🧪 2.3

1. RAG Agent "NLP " "Vector Database" 
2. Qdrant ( logistics, retail) Agent 

In [ ]:
# 🧪 : RAG Agent
# await demo_chat(rag_agent, [
# 'NLP ',
# 'Qdrant ',
# ])

---
## 👥 Section 2.4: Multi-Agent System

### Multi-Agent?

Agent → 
Multi-Agent: **** → 

### 🧩 3 Workflow ADK

| Pattern | | ? |
|---------|--------|-------------|
| **Sequential** | A→B→C | →→ |
| **Parallel** | A‖B‖C | 3 |
| **Loop** | | →→→... |

```
Sequential: [A] → [B] → [C] 

Parallel: [A] ─┐
 [B] ─┤─→ 
 [C] ─┘

Loop: [A] → [B] →╮ 
 ╰─────────╯ 
```

### 1️⃣ SequentialAgent — 

```
 → → 
 Step 1 Step 2 Step 3
```

 agent agent **shared state**

In [ ]:
%%time
from google.adk.agents import SequentialAgent

# Sub-agent 1: 
step1_search = Agent(
 name='step1_search',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction='',
 tools=[search_thai_ai_knowledge]
)

# Sub-agent 2: 
step2_summary = Agent(
 name='step2_summary',
 model='gemini-2.5-flash',
 description='',
 instruction=' step 3 bullet points '
)

# Sub-agent 3: 
step3_translate = Agent(
 name='step3_translate',
 model='gemini-2.5-flash',
 description='',
 instruction=' '
)

# SequentialAgent: → → 
sequential_pipeline = SequentialAgent(
 name='search_summarize_translate',
 description='Pipeline: → → ',
 sub_agents=[step1_search, step2_summary, step3_translate]
)

print('✅ SequentialAgent: → → ')
print(f' : {[a.name for a in sequential_pipeline.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=sequential_pipeline, app_name=sequential_pipeline.name)
await runner.session_service.create_session(app_name=sequential_pipeline.name, user_id='user_1', session_id='test_session')
# ─── Sequential Pipeline ───
print('═' * 60)
print('🧪 SequentialAgent: → → ')
print('═' * 60)

response = await chat_with_agent(runner, 'AI ', session_id='test_session')
print(f'\n📋 ( 3 ):')
print(response)

### 💡 
- **** step3_translate
- sub_agents → !
- **Shared State**: agent agent 

### 2️⃣ ParallelAgent — 

| Input | Task | Output |
|:-----:|------|:------:|
| | → healthcare | |
| | → banking | |
| | → education | |

** Sequential** 3 

In [ ]:
%%time
from google.adk.agents import ParallelAgent

# 3 agents 
healthcare_searcher = Agent(
 name='healthcare_searcher',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=' AI 2-3 ',
 tools=[search_thai_ai_knowledge]
)

banking_searcher = Agent(
 name='banking_searcher',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=' AI 2-3 ',
 tools=[search_thai_ai_knowledge]
)

education_searcher = Agent(
 name='education_searcher',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=' AI 2-3 ',
 tools=[search_thai_ai_knowledge]
)

# ParallelAgent: 3 !
parallel_search = ParallelAgent(
 name='parallel_multi_topic_search',
 description=' 3 ',
 sub_agents=[healthcare_searcher, banking_searcher, education_searcher]
)

print('✅ ParallelAgent: 3 ')
print(f' Agents: {[a.name for a in parallel_search.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=parallel_search, app_name=parallel_search.name)
await runner.session_service.create_session(app_name=parallel_search.name, user_id='user_1', session_id='test_session')
# ─── Parallel Search ───
print('═' * 60)
print('🧪 ParallelAgent: 3 ')
print('═' * 60)

response = await chat_with_agent(runner, ' AI ', session_id='test_session')
print(f'\n📋 ( 3 ):')
print(response)

### 💡 
- 3 agents **** → Sequential 3 ()
- agent **** 
- **** 

### 3️⃣ LoopAgent — 

| Step | Action | Result |
|:----:|--------|--------|
| 1 | | draft |
| 2 | | ? → Step 1 |
| 3 | | ✅ |

** self-improvement**: Agent → Agent → 

> ⚠️ `max_iterations` !

In [ ]:
%%time
from google.adk.agents import LoopAgent

# Agent 1: 
writer_agent = Agent(
 name='writer_agent',
 model='gemini-2.5-flash',
 description='',
 instruction=""" 
 feedback """,
 tools=[search_thai_ai_knowledge]
)

# Agent 2: (critic)
critic_agent = Agent(
 name='critic_agent',
 model='gemini-2.5-flash',
 description=' feedback',
 instruction=""":
 - ?
 - ?
 - ?
 
 → 
 → 'APPROVED: []'
 """
)

# LoopAgent: → → → → ... ( 3 )
loop_refiner = LoopAgent(
 name='refine_loop',
 description='',
 sub_agents=[writer_agent, critic_agent],
 max_iterations=3 # !
)

print('✅ LoopAgent: ↔ ( 3 )')
print(f' Max iterations: {loop_refiner.max_iterations}')

In [ ]:
runner = InMemoryRunner(agent=loop_refiner, app_name=loop_refiner.name)
await runner.session_service.create_session(app_name=loop_refiner.name, user_id='user_1', session_id='test_session')
# ─── Loop Agent ───
print('═' * 60)
print('🧪 LoopAgent: → → → ...')
print('═' * 60)

response = await chat_with_agent(runner, ' RAG ', session_id='test_session')
print(f'\n📋 ():')
print(response)

### 💡 
- Writer → Critic → Critic feedback → Writer 
- LoopAgent : (1) Critic APPROVED (2) `max_iterations`
- ** → ** 

> ⚠️ `max_iterations` → !

### 📊 3 + LLM-based Routing

| | Class | Flow | LLM ? | |
|--------|-------|------|---------------|----------|
| **Sequential** | `SequentialAgent` | A→B→C | ❌ Deterministic | →→ |
| **Parallel** | `ParallelAgent` | A‖B‖C | ❌ Deterministic | 3 |
| **Loop** | `LoopAgent` | A↔B () | ❌ Deterministic | ↔ |
| **LLM Routing** | `Agent` + `sub_agents` | LLM | ✅ LLM | Orchestrator |

> 💡 **Workflow Agents** (Sequential/Parallel/Loop) **deterministic** = 
> **LLM Routing** (Agent + sub_agents) LLM = 

### 4️⃣ LLM-based Routing — Agent 

 `Agent` + `sub_agents` → **LLM ** 

In [ ]:
%%time
# ─── LLM-based Routing: Orchestrator ───

# Sub-agents 
search_agent = Agent(
 name='search_agent',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=' ',
 tools=[search_thai_ai_knowledge]
)

summarizer_agent = Agent(
 name='summarizer_agent',
 model='gemini-2.5-flash',
 description=' ',
 instruction=' bullet points 3-5 emoji'
)

translator_agent = Agent(
 name='translator_agent',
 model='gemini-2.5-flash',
 description='',
 instruction=' '
)

# Orchestrator: LLM 
orchestrator = Agent(
 name='thai_ai_orchestrator',
 model='gemini-2.5-flash',
 description='Agent ',
 instruction=""" Agent :
 1. search_agent — 
 2. summarizer_agent — 
 3. translator_agent — 
 """,
 sub_agents=[search_agent, summarizer_agent, translator_agent]
)

print('✅ LLM Routing Orchestrator')

In [ ]:
# ─── LLM Routing ───
print('═' * 60)
print('🧪 LLM Routing: Agent ')
print('═' * 60)

await demo_chat(orchestrator, [
 'AI ',
 ' ',
 '',
])

### 💡 
- Orchestrator **** → **LLM ** description
- "..." → 
- "..." → 
- "..." → 

> ⚡ ****: agent 
> ⚠️ ****: LLM 

### 🧪 2.4

1. `SequentialAgent` : → → quiz
2. `ParallelAgent` AI vs vs 
3. `LoopAgent` agent 

In [ ]:
# 🧪 : pattern 
# Hint: SequentialAgent, ParallelAgent, LoopAgent
# from google.adk.agents import SequentialAgent # etc.


---
## 🧠 Section 2.5: Session & Memory

### Agent — Session

```
 Session: ()
 Session: → 
```

### Session ?

| | | |
|-----------|--------|--------|
| **History** | user ↔ agent | ": AI " → ": AI ..." |
| **State** | turn | , preferences, tool |
| **user_id** | | "student_001" |
| **session_id** | | "session_abc123" |

### SessionService ?

| | | | |
|------|--------|-----------|--------|
| `InMemorySessionService` | RAM | ❌ | Dev/ |
| `DatabaseSessionService` | DB | ✅ | Production |

> 💡 workshop `InMemorySessionService` 

In [ ]:
# ─── Demo: Agent ───
async def conversation_with_memory(agent, messages):
 """ session → Agent """
 runner = InMemoryRunner(agent=agent, app_name=agent.name)
 
 await runner.session_service.create_session(
 app_name=agent.name,
 user_id='student_demo',
 session_id='memory_demo'
 )

 for msg in messages:
 print(f'\n👤 : {msg}')
 content = types.Content(
 role='user',
 parts=[types.Part.from_text(text=msg)]
 )

 final_response = ''
 async for event in runner.run_async(
 user_id='student_demo',
 session_id='memory_demo',
 new_message=content
 ):
 if event.content and event.content.parts:
 for part in event.content.parts:
 if part.text:
 final_response = part.text
 
 print(f'🤖 Agent: {final_response}')

# 
await conversation_with_memory(greeting_agent, [
 ' ',
 ' AI ',
 '',
 '',
 '?',
])

### 💡 
- "" → Agent **** AI 
- "" → Agent **** healthcare + banking
- `chat_with_agent` (session ) → Agent 

> **Session = **: session → Agent 

---
## 🚀 Section 2.6: Agentic RAG ⭐

: **Agent + RAG Tool + Multi-Agent + Memory**

| Component | Role |
|---|---|
| 🎯 **Study Assistant** | Orchestrator Agent |
| 🔍 Search Agent (Qdrant) | |
| 📝 Summarizer Agent | |
| 🌐 Translator Agent | |
| 💾 Session | |

In [ ]:
# ─── Study Assistant ───

study_assistant = Agent(
 name='study_assistant',
 model='gemini-2.5-flash',
 description=' AI/RAG ',
 instruction=""" AI/RAG 

 :
 1. search_agent — ( AI/RAG )
 2. summarizer_agent — 
 3. translator_agent — 

 :
 - AI → search_agent → 
 - "..." 
 - emoji
 - 💡 
 """,
 sub_agents=[
 Agent(
 name='study_search_agent',
 model='gemini-2.5-flash',
 description=' AI ',
 instruction=' ',
 tools=[search_thai_ai_knowledge]
 ),
 Agent(
 name='study_summarizer_agent',
 model='gemini-2.5-flash',
 description=' ',
 instruction=' bullet points 3-5 emoji'
 ),
 Agent(
 name='study_translator_agent',
 model='gemini-2.5-flash',
 description='-',
 instruction=''
 ),
 ]
)

print('✅ Study Assistant !')

In [ ]:
# ─── Demo: Agentic RAG ───
print('═' * 60)
print('🎓 Demo: Study Assistant (Agentic RAG )')
print('═' * 60)

await conversation_with_memory(study_assistant, [
 ' RAG ',
 '',
 '',
 '',
])

### 🧪 2.6

1. Study Assistant 
2. Agent sub-agent 
3. tool quiz_generator

In [ ]:
# 🧪 : Study Assistant
# await conversation_with_memory(study_assistant, [
# 'embedding model ',
# ' multilingual',
# ' ',
# ])

---
## 🎯 : 

| Section | | |
|---------|------------|----------|
| 2.1 | Agent = LLM + Instructions | `google.adk.agents.Agent` |
| 2.2 | Tool = Python function Agent | `tools=[function]` |
| 2.3 | RAG Tool = VectorDB | Qdrant + Embedding |
| 2.4 | Multi-Agent = Agent | `sub_agents=[]` |
| 2.5 | Session = Agent | `InMemorySessionService` |
| 2.6 | Agentic RAG = | |

### 🛣️ Day 3: Evaluation
- RAG RAGAS
- Automated Testing
- Agent 

---

**🙏 !**